Additional End of week Exercise - week 2
Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
base_url = os.getenv("OPENROUTER_BASE_URL")
client = OpenAI(api_key=api_key, base_url=base_url)

In [ ]:
SYSTEM_PROMPT = """You are an expert technical Q&A assistant. You help with:
- Programming (Python, JavaScript, APIs, etc.)
- Debugging and error messages
- Best practices and design patterns
- Explaining code and technical concepts clearly

Respond in markdown when useful (code blocks, lists). Be concise but thorough.
If the user asks for code, provide runnable examples. If they mention a tool (e.g. current time), you may request it."""

In [ ]:
from datetime import datetime

def get_current_time() -> str:
    """Return current date and time. Use when the user asks for time or date."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time. Use when the user asks what time it is, or for the date.",
            "parameters": {"type": "object", "properties": {}},
        },
    }
]

In [ ]:
def execute_tool(name: str) -> str:
    if name == "get_current_time":
        return get_current_time()
    return "Unknown tool"

def _ensure_dict(msg):
    """Ensure message is a plain dict with only role and content (str)."""
    if isinstance(msg, dict) and "role" in msg and "content" in msg:
        return {"role": msg["role"], "content": str(msg.get("content") or "")}
    if hasattr(msg, "model_dump"):
        msg = msg.model_dump()
    return {"role": str(getattr(msg, "role", msg.get("role", "user"))), "content": str(getattr(msg, "content", msg.get("content")) or "")}

def chat_streaming(history, model_choice, stream=True):
    """Yields history + [{"role": "assistant", "content": accumulated}] like cwait sample."""
    history = list(history) if history else []
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + [_ensure_dict(h) for h in history]
    model_id = "gpt-4o-mini" if model_choice == "GPT" else "llama3.2:1b"
    if model_choice == "Ollama":
        api_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    else:
        api_client = client
    use_tools = model_choice != "Ollama" and not stream
    kwargs = {"model": model_id, "messages": messages, "stream": stream}
    if use_tools:
        kwargs["tools"] = TOOLS
        kwargs["tool_choice"] = "auto"
    if stream:
        stream_obj = api_client.chat.completions.create(**kwargs)
        accumulated = ""
        for chunk in stream_obj:
            if chunk.choices and chunk.choices[0].delta.content:
                accumulated += chunk.choices[0].delta.content
                yield history + [{"role": "assistant", "content": accumulated}]
        if accumulated:
            yield history + [{"role": "assistant", "content": accumulated}]
        return
    response = api_client.chat.completions.create(**kwargs)
    msg = response.choices[0].message
    while getattr(msg, "tool_calls", None):
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments or "{}"}} for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            result = execute_tool(tc.function.name)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        kwargs["messages"] = messages
        response = api_client.chat.completions.create(**kwargs)
        msg = response.choices[0].message
    final_content = msg.content or ""
    if not final_content:
        yield history + [{"role": "assistant", "content": ""}]
    else:
        for i in range(1, len(final_content) + 1):
            yield history + [{"role": "assistant", "content": final_content[:i]}]

In [ ]:
# Single handler: add user message to history, then stream (Reference: my TA solisoma)
def add_user_and_chat(message, history, model_choice, stream):
    """Add user message to history and yield (updated_history, None) for each chunk."""
    if not message or not message.strip():
        return
    history = list(history or [])
    history = [_ensure_dict(h) for h in history] + [{"role": "user", "content": message.strip()}]
    for updated in chat_streaming(history, model_choice, stream):
        yield updated

In [ ]:
# Gradio UI (cwait-style: one handler yields history updates, then clear message)
with gr.Blocks(title="Technical Q&A Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Technical Q&A Assistant\nAsk programming and technical questions. Supports **streaming**, **model switch**, and **tool use** (e.g. ask *What time is it?*).")
    chatbot = gr.Chatbot(height=400, type="messages", label="Chat")
    with gr.Row():
        message = gr.Textbox(placeholder="Type your question...", label="Message", scale=4)
        submit_btn = gr.Button("Send", variant="primary", scale=1)
    with gr.Row():
        model_dropdown = gr.Dropdown(choices=["GPT", "Ollama"], value="GPT", label="Model")
        stream_cb = gr.Checkbox(value=True, label="Stream response")

    def clear_msg():
        return ""

    submit_btn.click(
        fn=add_user_and_chat,
        inputs=[message, chatbot, model_dropdown, stream_cb],
        outputs=chatbot,
    ).then(fn=clear_msg, inputs=None, outputs=message)
    message.submit(
        fn=add_user_and_chat,
        inputs=[message, chatbot, model_dropdown, stream_cb],
        outputs=chatbot,
    ).then(fn=clear_msg, inputs=None, outputs=message)

In [ ]:
# Launch (queue required for streaming — from cwait sample)
demo.queue()
demo.launch()